# P5_Python_cientifico

Todos los datos son **simulados**. Lee primero las tres notas del día. Completa una celda y ejecuta su comprobación; esta fallará hasta resolver el ejercicio. Resolver aquí cuenta como los ejercicios de la nota, no duplica el trabajo.

La primera celda de código contiene los generadores pertinentes de P.D, idénticos a la nota. P1–P3: solo serie_anual. P4: añade serie_mensual. P5: añade malla. Las funciones importan dependencias al llamarse. Si faltan bibliotecas ejecuta antes la instalación opcional.

Ejecuta en un directorio de práctica: se crean CSV y NetCDF allí. No se necesitan archivos de datos externos.

In [ ]:
def serie_anual(n=24, tasa=0.02, nivel=14.0):
    """Serie simulada sin ruido; un valor por año desde 2000, en °C."""
    valores = []  # Lista vacía: iremos añadiendo valores.
    for i in range(n):
        valores.append(nivel + tasa * i)
    return valores

def serie_mensual(semilla=2026, faltantes=False):
    """Tres regiones simuladas, mensuales, 2000–2023; jamás datos NASA."""
    import numpy as np
    import pandas as pd
    rng = np.random.default_rng(semilla)
    fechas = pd.date_range("2000-01-01", periods=288, freq="MS")
    t = np.arange(288) / 12  # Tiempo en años desde el inicio.
    tablas = []
    for region, tasa in zip(["Costa", "Sierra", "Llanura"], [0.02, 0.04, -0.01]):
        estacion = 2 * np.sin(2 * np.pi * t)
        variacion = 0.12 * np.sin(2 * np.pi * t / 5)
        ruido = rng.normal(0, 0.08, len(t))
        valor = 14 + tasa * t + estacion + variacion + ruido
        if faltantes:
            valor[150::53] = np.nan  # Base 2000–2009 completa.
        tablas.append(pd.DataFrame({"fecha": fechas, "region": region,
                                   "temp_C": valor}))
    return pd.concat(tablas, ignore_index=True)

def malla(semilla=2026):
    """Dataset simulado mensual: time=24, lat=3, lon=4; temperatura en °C."""
    import numpy as np
    import pandas as pd
    import xarray as xr
    rng = np.random.default_rng(semilla)
    tiempo = pd.date_range("2000-01-01", periods=24, freq="MS")
    lat = np.array([-30.0, 0.0, 30.0])
    lon = np.array([-120.0, -90.0, -60.0, -30.0])
    t = np.arange(24).reshape(24, 1, 1) / 12
    tasas = np.array([[-0.03, -0.01, 0.01, 0.03]] * 3)
    valor = 14 + t * tasas + 0.3 * np.sin(2 * np.pi * t)
    valor = valor + rng.normal(0, 0.01, (24, 3, 4))
    ds = xr.Dataset({"temp": (("time", "lat", "lon"), valor)},
                    coords={"time": tiempo, "lat": lat, "lon": lon})
    ds.attrs["origen"] = "simulado"
    ds["temp"].attrs["units"] = "degC"
    ds["lat"].attrs["units"] = "degrees_north"
    ds["lon"].attrs["units"] = "degrees_east"
    return ds

## Instalación opcional para Colab

%pip instala en el entorno del notebook. Ejecuta una vez antes de usar bibliotecas; reinicia si cambiaste versiones. En local usa venv y requirements.txt.

In [ ]:
%pip install -q "numpy>=2,<3" "pandas>=2.2,<3" matplotlib scipy xarray netCDF4 statsmodels pymannkendall pytest

## Núcleo · 45 min

### P5.E-01 · ⭐ · Predice la salida · 5 min · Leer una pendiente

**Datos simulados.** Ajusta datos simulados y guarda decada.

<details><summary>Pista</summary>

slope tiene unidades por año.

</details>

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import linregress, theilslopes, kendalltau
x=np.arange(5)
y=14+0.02*x
# TU CÓDIGO AQUÍ

In [ ]:
assert (abs(decada-0.2)<1e-10), "Leer una pendiente: revisa valores, unidades y forma"

### P5.E-02 · ⭐ · Completa el código · 5 min · Selección cercana

**Datos simulados.** Elige celda lat=2,lon=-88 con nearest en malla de P.D.

<details><summary>Pista</summary>

Busca las coordenadas disponibles.

</details>

In [ ]:
ds=malla()
# TU CÓDIGO AQUÍ

In [ ]:
assert (float(celda.lat)==0 and float(celda.lon)==-90 and celda.sizes['time']==24), "Selección cercana: revisa valores, unidades y forma"

### P5.E-03 · ⭐⭐ · Corrige el error · 10 min · Años reales al ajustar

**Datos simulados.** Falta 2002: ajusta con años, no posiciones; guarda r.

<details><summary>Pista</summary>

Traslada el origen sin comprimir huecos.

</details>

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import linregress, theilslopes, kendalltau
x=np.array([2000,2001,2003,2004])
y=14+0.02*(x-2000)
# TU CÓDIGO AQUÍ

In [ ]:
assert (abs(r.slope-0.02)<1e-10 and abs(r.intercept-14)<1e-10), "Años reales al ajustar: revisa valores, unidades y forma"

### P5.E-04 · ⭐⭐ · Escribe desde cero · 10 min · Contrato reproducible

**Datos simulados.** Define tasa_decada(x,y) con máscara común; raise si faltan tres pares o años distintos.

<details><summary>Pista</summary>

Filtra ambos arreglos juntos.

</details>

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import linregress, theilslopes, kendalltau
# TU CÓDIGO AQUÍ

In [ ]:
assert (abs(tasa_decada([2000,2001,2002,2003],[14,14.02,np.nan,14.06])-0.2)<1e-10), "Contrato reproducible: revisa valores, unidades y forma"

### P5.E-05 · ⭐⭐⭐ · Reto Tierra · 15 min · Integración núcleo P5

**Datos simulados.** Con malla P.D calcula mapa medio temporal y serie regional lat -30 a 0; escribe NetCDF y lee copia con load.

<details><summary>Pista</summary>

identical compara valores, coordenadas y atributos.

</details>

In [ ]:
import xarray as xr
ds=malla()
# TU CÓDIGO AQUÍ

In [ ]:
assert (mapa.shape==(3,4) and serie.shape==(24,) and copia.identical(ds)), "Integración núcleo P5: revisa valores, unidades y forma"

## Ampliación · 60 min

### P5.E-06 · ⭐⭐⭐ · Escribe desde cero · 20 min · Resultados estadísticos ampliados

**Datos simulados.** Con P.D anual lee Mann-Kendall y OLS. Guarda tasa_mk,tasa_ols por década y texto_resumen. Los pasos son años regulares.

<details><summary>Pista</summary>

El coeficiente 1 corresponde a x tras añadir constante.

</details>

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import linregress, theilslopes, kendalltau
import pymannkendall as mk
import statsmodels.api as sm
y=np.array(serie_anual())
x=np.arange(len(y))
# TU CÓDIGO AQUÍ

In [ ]:
assert (abs(tasa_mk-0.2)<1e-8 and abs(tasa_ols-0.2)<1e-8 and 'OLS Regression Results' in texto_resumen), "Resultados estadísticos ampliados: revisa valores, unidades y forma"

### P5.E-07 · ⭐⭐⭐ · Escribe desde cero · 20 min · xarray ponderado y anual

**Datos simulados.** Promedia malla con pesos cos(lat), luego resample YE; conserva serie,anual.

<details><summary>Pista</summary>

Los pesos se alinean con lat.

</details>

In [ ]:
import numpy as np
ds=malla()
# TU CÓDIGO AQUÍ

In [ ]:
assert (serie.sizes['time']==24 and anual.sizes['time']==2 and bool(serie.notnull().all())), "xarray ponderado y anual: revisa valores, unidades y forma"

### P5.E-08 · ⭐⭐⭐ · Reto Tierra · 20 min · Integración ampliada P5

**Datos simulados.** Escribe test_conversion.py con test_decada que compruebe la conversión 0.02→0.2; mide una resta vectorizada de P.D en duracion. Ejecuta pytest en la terminal de práctica.

<details><summary>Pista</summary>

pytest descubre funciones test_.

</details>

In [ ]:
import numpy as np
from pathlib import Path
from time import perf_counter
# TU CÓDIGO AQUÍ

In [ ]:
assert (Path('test_conversion.py').exists() and duracion>=0 and np.allclose(anom,np.arange(24)*0.02)), "Integración ampliada P5: revisa valores, unidades y forma"